# 🎙️ Automated Meeting Minutes from Audio

[![HuggingFace](https://img.shields.io/badge/🤗-HuggingFace-yellow)](https://huggingface.co)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

This notebook demonstrates an end-to-end pipeline that:
1. **Transcribes** an audio recording using either open-source (Whisper via HuggingFace) or OpenAI's API
2. **Summarizes** the transcription into structured meeting minutes using a quantized LLaMA model

---

## 📋 Pipeline Overview

```
Audio File (.mp3)
      │
      ▼
┌─────────────────────────────┐
│  STEP 1: Transcription      │
│  Option A: Whisper (OSS)    │
│  Option B: OpenAI API       │
└────────────┬────────────────┘
             │
             ▼
┌─────────────────────────────┐
│  STEP 2: Summarization      │
│  LLaMA 3.2 (4-bit quant)    │
│  → Structured Markdown      │
└─────────────────────────────┘
```

## 📁 Dataset

This example uses an extract from Denver City Council meeting audio, sourced from the [MeetingBank dataset](https://huggingface.co/datasets/huuuyeah/meetingbank) on HuggingFace.

- **Download the audio extract used here:** [denver_extract.mp3](https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing)
- **Full audio dataset:** [MeetingBank Audio](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main)

> 💡 **Tip:** You can swap in your own `.mp3` file to generate minutes from any meeting recording!

## 🔧 Requirements

| Component | Requirement |
|-----------|-------------|
| Runtime | Google Colab T4 GPU |
| Secrets | `HF_TOKEN`, `OPENAI_API_KEY` (optional) |
| Drive file | `MyDrive/llms/denver_extract.mp3` |

---

## ⚠️ Colab Runtime Troubleshooting

If you encounter this error mid-run:

> `Runtime error: CUDA is required but not available for bitsandbytes`

This is **not** a package version issue — it means Google Colab swapped your runtime. Fix it by:

1. **Kernel** → *Disconnect and delete runtime*
2. Reload the notebook → **Edit** → *Clear All Outputs*
3. Reconnect to a **T4 GPU** runtime
4. Confirm GPU via **View Resources** (top-right menu)
5. Re-run all cells from top

---
## 📦 Install Dependencies

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

## 🔑 Imports & Configuration

In [ ]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Model & File Configuration ──────────────────────────────────────────────

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"   # Summarization model
WHISPER = "openai/whisper-medium.en"          # Open-source transcription model
AUDIO_MODEL = "gpt-4o-mini-transcribe"        # OpenAI transcription model (Option B)

AUDIO_DRIVE_PATH = "/content/drive/MyDrive/llms/denver_extract.mp3"

## 📂 Mount Google Drive & Load Audio

In [ ]:
# Mount Google Drive — authorize when prompted
drive.mount("/content/drive")

audio_filename = AUDIO_DRIVE_PATH
assert os.path.exists(audio_filename), (
    f"Audio file not found at {audio_filename}.\n"
    "Please download it from the link in the introduction and place it at MyDrive/llms/denver_extract.mp3"
)
print(f"✅ Audio file found: {audio_filename}")

In [ ]:
# Authenticate with HuggingFace Hub (needed for gated models like LLaMA)
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
print("✅ HuggingFace login successful")

# Open audio file (needed for OpenAI transcription option)
audio_file = open(audio_filename, "rb")

---
# STEP 1: Transcribe Audio

Choose **Option A** (open-source, free) or **Option B** (OpenAI API, faster). Both produce equivalent results.

## Option A: Open-Source Transcription with Whisper (HuggingFace)

Uses [`openai/whisper-medium.en`](https://huggingface.co/openai/whisper-medium.en) via the HuggingFace `transformers` pipeline — **no API key required**.

In [ ]:
from transformers import pipeline

print("Loading Whisper pipeline...")
pipe = pipeline(
    "automatic-speech-recognition",
    model=WHISPER,
    dtype=torch.float16,
    device="cuda",
    return_timestamps=True
)

print("Transcribing audio...")
result = pipe(audio_filename)
open_source_transcription = result["text"]

print("\n── Whisper Transcription ──")
print(open_source_transcription)

## Option B: Transcription with OpenAI API

Uses `gpt-4o-mini-transcribe` — requires an `OPENAI_API_KEY` secret in Colab.

In [ ]:
openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

print("Transcribing with OpenAI API...")
openai_transcription = openai.audio.transcriptions.create(
    model=AUDIO_MODEL,
    file=audio_file,
    response_format="text"
)

print("\n── OpenAI Transcription ──")
print(openai_transcription)

### 🔍 Compare Transcription Outputs

In [ ]:
print("=" * 60)
print("OPTION A — Whisper (Open Source)")
print("=" * 60)
display(Markdown(open_source_transcription))

print("\n" + "=" * 60)
print("OPTION B — OpenAI API")
print("=" * 60)
display(Markdown(openai_transcription))

In [ ]:
# ── Select which transcription to use for meeting minutes ───────────────────
# Change this to `openai_transcription` if you used Option B
transcription = open_source_transcription

---
# STEP 2: Generate Meeting Minutes with LLaMA

The transcription is fed to a **4-bit quantized LLaMA 3.2 3B Instruct** model to produce structured meeting minutes in Markdown.

**Why 4-bit quantization?**  
LLaMA 3.2 3B at full precision requires ~12 GB VRAM. With `bitsandbytes` 4-bit NF4 quantization, it fits comfortably in the free T4's 15 GB.

In [ ]:
# ── Prompt Construction ──────────────────────────────────────────────────────

system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user",   "content": user_prompt}
]

In [ ]:
# ── 4-bit Quantization Config ────────────────────────────────────────────────

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,   # nested quantization for extra memory savings
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"         # NormalFloat4 — best quality at 4-bit
)

In [ ]:
# ── Load Model & Generate ────────────────────────────────────────────────────

print(f"Loading tokenizer: {LLAMA}")
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token

inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)

print(f"Loading model with 4-bit quantization: {LLAMA}")
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config
)

print("\n── Generating Meeting Minutes ──\n")
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

## 📄 Render Final Meeting Minutes

In [ ]:
response = tokenizer.decode(outputs[0])

# Extract only the assistant's response (strip prompt tokens)
if "<|start_header_id|>assistant<|end_header_id|>" in response:
    minutes = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    minutes = minutes.replace("<|eot_id|>", "").strip()
else:
    minutes = response

display(Markdown(minutes))

---
## 💾 (Optional) Save Minutes to File

In [ ]:
output_path = "/content/drive/MyDrive/llms/meeting_minutes_output.md"

with open(output_path, "w") as f:
    f.write(minutes)

print(f"✅ Meeting minutes saved to: {output_path}")

---
## 🧠 Key Concepts Demonstrated

| Concept | Description |
|---------|-------------|
| **ASR Pipeline** | Automatic speech recognition via HuggingFace `pipeline` API |
| **Whisper** | OpenAI's open-source speech-to-text model (medium.en variant) |
| **4-bit Quantization** | `bitsandbytes` NF4 double-quant to run LLaMA on free-tier GPU |
| **LLaMA 3.2 Instruct** | Meta's instruction-tuned model for structured text generation |
| **Chat Templating** | `apply_chat_template` for correct system/user message formatting |
| **Token Streaming** | `TextStreamer` for real-time output display during generation |

---
## 🔗 References & Credits

- [MeetingBank Dataset (HuggingFace)](https://huggingface.co/datasets/huuuyeah/meetingbank)
- [Whisper Medium EN (HuggingFace)](https://huggingface.co/openai/whisper-medium.en)
- [LLaMA 3.2 3B Instruct (HuggingFace)](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
- [bitsandbytes Quantization Docs](https://huggingface.co/docs/transformers/quantization/bitsandbytes)
- Extended streaming variant using `TextIteratorStreamer` + Gradio UI by [Emad S.](https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D)